## Task 1 —> Data Profiling & Duplicate Detection

1. Load the Social Media Sentiment dataset and display the number of rows and columns.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("Social_Media_Sentiment_Dataset.csv")

print("Shape:", df.shape)
df.head()


2. Shows column names, data types, and non-null counts all at once.

In [ ]:
df.info()


3. Find duplicate rows: `.duplicated()` checks whether a complete row is repeated. `.sum()` counts the duplicate rows.

In [ ]:
print("Exact duplicate rows:", df.duplicated().sum())


4. Check duplicate social-media posts using the main text field. This identifies repeated post text even when other metadata may differ.

In [ ]:
post_duplicates = df.duplicated(subset=["Text"]).sum()
print("Duplicate posts by text:", post_duplicates)


5. Remove unnecessary index columns. `Unnamed: 0.1` and `Unnamed: 0` are stored index columns rather than analytical features.

In [ ]:
df = df.drop(columns=["Unnamed: 0.1", "Unnamed: 0"], errors="ignore")
print("Shape after removing index columns:", df.shape)


## Task 2 — Missing Values & Data Quality

6. Create a missing-value report showing both the number and percentage of missing values in each column.

In [ ]:
missing_count = df.isnull().sum()

missing_percent = (
    df.isnull().sum() / len(df)
) * 100

missing_report = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing %": missing_percent.round(2)
})

missing_report = missing_report[
    missing_report["Missing Count"] > 0
].sort_values(
    "Missing Count",
    ascending=False
)

print(missing_report)

if missing_report.empty:
    print("No missing values found.")


7. Check the unique values of the main categorical fields before cleaning. This helps identify extra spaces or inconsistent labels.

In [ ]:
print("Raw Platform values:")
print(df["Platform"].unique())

print("\nNumber of raw Sentiment labels:", df["Sentiment"].nunique())


8. Check data types of all columns.

In [ ]:
print(df.dtypes)


## Task 3 —> Data Cleaning

9. Remove extra spaces from all text/category columns. This is important because values such as `Twitter` and `Twitter ` should be treated as the same platform.

In [ ]:
text_columns = df.select_dtypes(include="object").columns

for column in text_columns:
    df[column] = df[column].str.strip()

print("Text columns cleaned successfully.")


10. Convert the Timestamp column into a proper datetime format.

In [ ]:
df["Timestamp"] = pd.to_datetime(
    df["Timestamp"],
    errors="coerce"
)

print("Timestamp datatype:", df["Timestamp"].dtype)


11. Create an Engagement column. Engagement is calculated as Likes + Retweets.

In [ ]:
df["Engagement"] = df["Likes"] + df["Retweets"]

print(df[["Likes", "Retweets", "Engagement"]].head())


12. Check the cleaned platform categories.

In [ ]:
print("Clean Platform values:")
print(df["Platform"].unique())

print("\nPlatform counts:")
print(df["Platform"].value_counts())


13. Check the cleaned sentiment categories. The dataset contains many emotion/sentiment labels, so the original labels are retained.

In [ ]:
print("Number of unique sentiment labels:", df["Sentiment"].nunique())
print("\nTop 15 sentiment labels:")
print(df["Sentiment"].value_counts().head(15))


## Task 4 — Statistical Analysis

14. Descriptive statistics for the numerical variables.

In [ ]:
df.describe()


15. Calculate important engagement statistics.

In [ ]:
print("Average Likes:", round(df["Likes"].mean(), 2))
print("Median Likes:", df["Likes"].median())
print("Average Retweets:", round(df["Retweets"].mean(), 2))
print("Median Retweets:", df["Retweets"].median())
print("Average Engagement:", round(df["Engagement"].mean(), 2))
print("Median Engagement:", df["Engagement"].median())


16. Display the minimum and maximum values for Likes, Retweets, and Engagement.

In [ ]:
print("Likes - Min:", df["Likes"].min(), "Max:", df["Likes"].max())
print("Retweets - Min:", df["Retweets"].min(), "Max:", df["Retweets"].max())
print("Engagement - Min:", df["Engagement"].min(), "Max:", df["Engagement"].max())


## Task 5 — Analyze Social Media Platforms

17. Count the number of posts available on each platform.

In [ ]:
platform_counts = df["Platform"].value_counts()

print(platform_counts)


18. Calculate the average Likes, Retweets, and Engagement for each platform.

In [ ]:
platform_summary = df.groupby("Platform")[[
    "Likes",
    "Retweets",
    "Engagement"
]].mean().round(2)

print(platform_summary)


## Task 6 — Visualization

19. Number of posts by social-media platform. This bar chart compares the number of records across platforms.

In [ ]:
platform_counts.plot(kind="bar")

plt.title("Number of Posts by Social Media Platform")
plt.xlabel("Platform")
plt.ylabel("Number of Posts")
plt.xticks(rotation=0)

plt.show()


20. Number of posts by year. This shows how the dataset is distributed across years.

In [ ]:
year_counts = df["Year"].value_counts().sort_index()

year_counts.plot(kind="line", marker="o")

plt.title("Number of Social Media Posts by Year")
plt.xlabel("Year")
plt.ylabel("Number of Posts")
plt.grid(True)

plt.show()


21. Distribution of Likes. This histogram shows how Likes are distributed in the dataset.

In [ ]:
df["Likes"].plot(
    kind="hist",
    bins=15
)

plt.title("Distribution of Likes")
plt.xlabel("Likes")
plt.ylabel("Number of Posts")

plt.show()


22. Distribution of Retweets.

In [ ]:
df["Retweets"].plot(
    kind="hist",
    bins=15
)

plt.title("Distribution of Retweets")
plt.xlabel("Retweets")
plt.ylabel("Number of Posts")

plt.show()


## Task 7 — Sentiment Analysis

23. Count the most frequent sentiment categories.

In [ ]:
sentiment_counts = df["Sentiment"].value_counts()

print(sentiment_counts.head(15))


24. Visualize the top 15 sentiment categories.

In [ ]:
sentiment_counts.head(15).plot(kind="bar")

plt.title("Top 15 Sentiment Categories")
plt.xlabel("Sentiment")
plt.ylabel("Number of Posts")
plt.xticks(rotation=45, ha="right")

plt.tight_layout()
plt.show()


25. Compare platforms across the top 10 sentiment categories.

In [ ]:
top_sentiments = df["Sentiment"].value_counts().head(10).index

sentiment_platform = df[
    df["Sentiment"].isin(top_sentiments)
]

platform_sentiment_table = pd.crosstab(
    sentiment_platform["Platform"],
    sentiment_platform["Sentiment"]
)

print(platform_sentiment_table)


26. Heatmap of Platform × Sentiment.

In [ ]:
plt.figure(figsize=(12, 6))

sns.heatmap(
    platform_sentiment_table,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.title("Platform vs Top 10 Sentiments")
plt.xlabel("Sentiment")
plt.ylabel("Platform")

plt.tight_layout()
plt.show()


## Task 8 — Engagement Analysis

27. Compare average Likes by platform.

In [ ]:
average_likes = (
    df.groupby("Platform")["Likes"]
    .mean()
    .sort_values(ascending=False)
)

print(average_likes)


28. Visualize average Likes by platform.

In [ ]:
average_likes.plot(kind="bar")

plt.title("Average Likes by Platform")
plt.xlabel("Platform")
plt.ylabel("Average Likes")
plt.xticks(rotation=0)

plt.show()


29. Compare average Retweets by platform.

In [ ]:
average_retweets = (
    df.groupby("Platform")["Retweets"]
    .mean()
    .sort_values(ascending=False)
)

print(average_retweets)


30. Visualize average Retweets by platform.

In [ ]:
average_retweets.plot(kind="bar")

plt.title("Average Retweets by Platform")
plt.xlabel("Platform")
plt.ylabel("Average Retweets")
plt.xticks(rotation=0)

plt.show()


31. Compare average Engagement by platform.

In [ ]:
average_engagement = (
    df.groupby("Platform")["Engagement"]
    .mean()
    .sort_values(ascending=False)
)

print(average_engagement)


32. Visualize average Engagement by platform.

In [ ]:
average_engagement.plot(kind="bar")

plt.title("Average Engagement by Platform")
plt.xlabel("Platform")
plt.ylabel("Average Engagement")
plt.xticks(rotation=0)

plt.show()


## Task 9 — Country & Time Analysis

33. Find the top 10 countries by number of posts.

In [ ]:
country_counts = df["Country"].value_counts().head(10)

print(country_counts)


34. Visualize the top 10 countries.

In [ ]:
country_counts.plot(kind="bar")

plt.title("Top 10 Countries by Number of Posts")
plt.xlabel("Country")
plt.ylabel("Number of Posts")
plt.xticks(rotation=45, ha="right")

plt.tight_layout()
plt.show()


35. Analyze posts by month.

In [ ]:
month_counts = df["Month"].value_counts().sort_index()

print(month_counts)


36. Visualize the monthly trend.

In [ ]:
month_counts.plot(
    kind="line",
    marker="o"
)

plt.title("Number of Posts by Month")
plt.xlabel("Month")
plt.ylabel("Number of Posts")
plt.xticks(range(1, 13))
plt.grid(True)

plt.show()


37. Analyze posts by hour.

In [ ]:
hour_counts = df["Hour"].value_counts().sort_index()

print(hour_counts)


38. Visualize hourly activity.

In [ ]:
hour_counts.plot(
    kind="line",
    marker="o"
)

plt.title("Number of Posts by Hour")
plt.xlabel("Hour")
plt.ylabel("Number of Posts")
plt.grid(True)

plt.show()


## Task 10 — Find Highly Engaging Posts

39. Sort posts by Engagement to identify highly engaging records.

In [ ]:
top_posts = df.sort_values(
    "Engagement",
    ascending=False
)

print(
    top_posts[
        ["Text", "Platform", "Sentiment", "Likes", "Retweets", "Engagement"]
    ].head(10)
)


40. Display the top 10 posts by Likes.

In [ ]:
top_liked = df.sort_values(
    "Likes",
    ascending=False
)

print(
    top_liked[
        ["Text", "Platform", "Likes", "Retweets", "Engagement"]
    ].head(10)
)


41. Display the top 10 posts by Retweets.

In [ ]:
top_retweeted = df.sort_values(
    "Retweets",
    ascending=False
)

print(
    top_retweeted[
        ["Text", "Platform", "Likes", "Retweets", "Engagement"]
    ].head(10)
)


42. Visualize the top 10 posts by Engagement.

In [ ]:
top_10_engagement = (
    df.nlargest(10, "Engagement")
    .sort_values("Engagement")
)

plt.figure(figsize=(9, 5))

plt.barh(
    range(len(top_10_engagement)),
    top_10_engagement["Engagement"]
)

plt.yticks(
    range(len(top_10_engagement)),
    [f"Post {i+1}" for i in range(len(top_10_engagement))]
)

plt.title("Top 10 Posts by Engagement")
plt.xlabel("Engagement")

plt.tight_layout()
plt.show()


## Task 11 — Correlation Analysis

43. Calculate correlations among Likes, Retweets, Engagement, and time-related numerical variables.

In [ ]:
numeric_columns = [
    "Retweets",
    "Likes",
    "Year",
    "Month",
    "Day",
    "Hour",
    "Engagement"
]

correlation = df[numeric_columns].corr()

print(correlation)


44. Visualize the correlation matrix using a heatmap.

In [ ]:
plt.figure(figsize=(10, 7))

sns.heatmap(
    correlation,
    annot=True,
    fmt=".2f",
    cmap="Blues"
)

plt.title("Correlation Heatmap")
plt.tight_layout()

plt.show()


## Task 12 — Hashtag Analysis

45. Extract and count the most frequently used hashtags.

In [ ]:
from collections import Counter

hashtags = []

for value in df["Hashtags"].dropna():
    for tag in str(value).split(","):
        tag = tag.strip()
        if tag:
            hashtags.append(tag)

top_hashtags = Counter(hashtags).most_common(15)

print("Top 15 Hashtags:")

for hashtag, count in top_hashtags:
    print(hashtag, ":", count)


46. Visualize the top 15 hashtags.

In [ ]:
hashtag_df = pd.DataFrame(
    top_hashtags,
    columns=["Hashtag", "Count"]
)

hashtag_df = hashtag_df.sort_values("Count")

plt.figure(figsize=(10, 6))

plt.barh(
    hashtag_df["Hashtag"],
    hashtag_df["Count"]
)

plt.title("Top 15 Hashtags")
plt.xlabel("Frequency")

plt.tight_layout()
plt.show()


## Task 13 — Final Summary

47. Display the main project statistics.

In [ ]:
print("========== SOCIAL MEDIA ANALYTICS SUMMARY ==========")

print("Total Posts:", len(df))
print("Total Likes:", int(df["Likes"].sum()))
print("Total Retweets:", int(df["Retweets"].sum()))
print("Total Engagement:", int(df["Engagement"].sum()))

print("Average Likes:", round(df["Likes"].mean(), 2))
print("Average Retweets:", round(df["Retweets"].mean(), 2))
print("Average Engagement:", round(df["Engagement"].mean(), 2))

print("Number of Platforms:", df["Platform"].nunique())
print("Number of Countries:", df["Country"].nunique())
print("Number of Sentiment Categories:", df["Sentiment"].nunique())


48. Display the final cleaned dataset.

In [ ]:
print("Final dataset shape:", df.shape)
df.head(10)


49. Save the cleaned dataset so it can be reused in Assignment 2 and Assignment 3.

In [ ]:
df.to_csv(
    "Clean_Social_Media_Sentiment_Dataset.csv",
    index=False
)

print("Clean dataset saved successfully.")


## Conclusion

In this assignment, the Social Media Sentiment dataset was successfully analyzed using Python, Pandas, NumPy, Matplotlib, and Seaborn. The dataset was first inspected to understand its structure, columns, data types, missing values, and duplicate records.

The data was then cleaned by removing unnecessary index columns, trimming extra spaces from categorical fields, converting the Timestamp column into datetime format, and creating an Engagement metric from Likes and Retweets.

The analysis covered platform distribution, sentiment categories, engagement, country-wise activity, yearly and monthly trends, hourly activity, hashtags, highly engaging posts, and correlations among numerical variables. Multiple charts and heatmaps were used to present the results clearly.

Overall, the assignment provides practical experience in data profiling, data cleaning, exploratory data analysis, statistical analysis, and data visualization using a real social-media dataset.

### Expected Dataset Facts

For the supplied CSV, the initial dataset contains **732 rows and 15 columns**. After removing the two unnecessary index columns and adding the Engagement column, the working dataset contains **732 rows and 14 columns**. The dataset has no missing values and no exact duplicate rows. The raw Sentiment field contains many labels, so the original categories are preserved rather than forcing them into only three classes.